In [ ]:
# Install required libraries
# Quiet installation of compatible and stable versions
!pip install -q transformers==4.46.0 datasets==3.1.0 torch accelerate

In [ ]:
# Import core libraries for tokenization, model loading, and training
import torch
from transformers import AutoTokenizer, DistilBertForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer
from datasets import load_dataset, DatasetDict
import math, os

# Confirm library versions
import transformers, datasets
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

Transformers: 4.46.0
Datasets: 3.1.0


In [ ]:
# Load DistilBERT tokenizer and masked-language-model head
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = DistilBertForMaskedLM.from_pretrained(model_name)

print("Tokenizer and model loaded successfully.")
print("Mask token:", tokenizer.mask_token)

Tokenizer and model loaded successfully.
Mask token: [MASK]


In [ ]:
# Load IMDb dataset; restrict to a small subset for quick training
raw = load_dataset("imdb")
train_small = raw["train"].shuffle(seed=42).select(range(2000))  # 2k samples for speed
test_small = raw["test"].shuffle(seed=42).select(range(500))     # 500 samples for eval
dataset = DatasetDict({"train": train_small, "test": test_small})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 500
    })
})


In [ ]:
# Tokenize text without padding; truncation keeps sequences within model limit
max_length = 128

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding=False)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text", "label"])
print("Tokenization complete.")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenization complete.


In [ ]:
# Combine tokens into uniform blocks of max_length for efficient MLM training
def group_texts(examples):
    concatenated = []
    for ids in examples["input_ids"]:
        concatenated.extend(ids)
    total_len = (len(concatenated) // max_length) * max_length
    result = {"input_ids": [concatenated[i:i+max_length] for i in range(0, total_len, max_length)]}
    result["attention_mask"] = [[1]*max_length for _ in result["input_ids"]]
    return result

tokenized_blocked = tokenized.map(
    group_texts, batched=True, batch_size=500, remove_columns=tokenized["train"].column_names
)
print("Token grouping complete.")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Token grouping complete.


In [ ]:
# Create a collator that randomly masks tokens (15%) during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)
print("Data collator ready.")

Data collator ready.


In [ ]:
# Configure training; use one-epoch loops externally for compatibility and control
training_args = TrainingArguments(
    output_dir="./mlm_results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=2
)

# Initialize Trainer for training and evaluation
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_blocked["train"],
    eval_dataset=tokenized_blocked["test"],
    data_collator=data_collator
)

print("Trainer initialized successfully.")

Trainer initialized successfully.


In [ ]:
# Run manual epoch loop for stable behavior across versions
NUM_EPOCHS = 3
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{NUM_EPOCHS} ===")
    train_result = trainer.train()
    if train_result.metrics:
        print("Train metrics:", train_result.metrics)

    eval_results = trainer.evaluate()
    loss = eval_results.get("eval_loss", None)
    if loss is not None:
        perplexity = math.exp(loss) if loss < 100 else float("inf")
        print(f"Eval loss: {loss:.4f} | Perplexity: {perplexity:.2f}")
    trainer.save_model(f"./mlm_results/epoch-{epoch}")
    tokenizer.save_pretrained(f"./mlm_results/epoch-{epoch}")
    print(f"Checkpoint saved for epoch {epoch}.")


=== Epoch 1/3 ===


Step,Training Loss
100,2.770500


Step,Training Loss
100,2.770500


In [2]:
# Required imports if previous cells were not run or kernel was reset
import torch
from transformers import AutoTokenizer, DistilBertForMaskedLM

# Load DistilBERT tokenizer and masked-language-model head if not already loaded
if 'tokenizer' not in locals() or 'model' not in locals():
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = DistilBertForMaskedLM.from_pretrained(model_name)


# Interactive prediction for user-provided masked sentences

# Ask user to input a sentence with one or more [MASK] tokens
user_text = input("Enter a sentence with [MASK] tokens (e.g., 'The movie was [MASK] and the story was [MASK].'): ")

# Tokenize input text
inputs = tokenizer(user_text, return_tensors="pt")

# Perform prediction without gradient computation
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Identify [MASK] token positions
mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)

# Predict top 5 words for each masked token
for idx in range(mask_positions[0].shape[0]):
    batch_i = mask_positions[0][idx].item()
    token_i = mask_positions[1][idx].item()
    topk = torch.topk(logits[batch_i, token_i], k=5).indices.tolist()
    preds = [tokenizer.decode([t]).strip() for t in topk]
    print(f"\nTop predictions for mask {idx+1}: {preds}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Enter a sentence with [MASK] tokens (e.g., 'The movie was [MASK] and the story was [MASK].'): The movie was [MASK] and the plot was [MASK].

Top predictions for mask 1: ['filmed', 'cancelled', 'banned', 'shot', 'released']

Top predictions for mask 2: ['changed', 'scrapped', 'reversed', 'altered', 'lost']
